In [21]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F
!pip install openai
import openai
import json

In [40]:
import json
from collections import defaultdict

match_id = 9045752693

with SparkSessionContext("case-3") as spark:

    data = spark.sql(f"""
        WITH events AS (
            SELECT
                CONCAT('kill:', CAST(id AS STRING)) AS event_id,
                'hero_kill' AS event_type,
                CAST(time AS INTEGER) AS event_time,
                CAST(steam_account_id AS BIGINT) AS steam_account_id,
                CAST(attacker AS INTEGER) AS attacker,
                CAST(target AS INTEGER) AS target,
                CAST(gold AS INTEGER) AS gold,
                CAST(xp AS INTEGER) AS xp,
                CAST(NULL AS INTEGER) AS cost,
                CAST(NULL AS INTEGER) AS radiant_mask,
                CAST(NULL AS INTEGER) AS dire_mask,
                CAST(NULL AS STRING) AS item_name,
                CAST(NULL AS INTEGER) AS item_cost,
                CAST(NULL AS INTEGER) AS streak_hero_id,
                CAST(NULL AS INTEGER) AS streak_value
            FROM lakehouse.raw.mongo__kill_events
            WHERE match_id = {match_id}
              AND id IS NOT NULL

            UNION ALL

            SELECT
                CONCAT('tower_destroyed:', CAST(id AS STRING)),
                'tower_destroyed',
                CAST(time AS INTEGER),
                CAST(NULL AS BIGINT),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(radiant AS INTEGER),
                CAST(dire AS INTEGER),
                CAST(NULL AS STRING),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER)
            FROM lakehouse.raw.mongo__tower_death_events
            WHERE match_id = {match_id}
              AND id IS NOT NULL

            UNION ALL

            SELECT
                CONCAT('roshan_reward:', CAST(id AS STRING)),
                'roshan_reward',
                CAST(time AS INTEGER),
                CAST(steam_account_id AS BIGINT),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(amount AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS STRING),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER)
            FROM lakehouse.raw.mongo__gold_events
            WHERE match_id = {match_id}
              AND id IS NOT NULL
              AND LOWER(reason) = 'roshan'

            UNION ALL

            SELECT
                CONCAT('buyback:', CAST(id AS STRING)),
                'buyback',
                CAST(time AS INTEGER),
                CAST(steam_account_id AS BIGINT),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(cost AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS STRING),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER)
            FROM lakehouse.raw.mongo__buyback_events
            WHERE match_id = {match_id}
              AND id IS NOT NULL

            UNION ALL

            SELECT
                CONCAT('purchase:', CAST(p.id AS STRING)),
                'item_purchase',
                CAST(p.time AS INTEGER),
                CAST(p.steam_account_id AS BIGINT),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                COALESCE(i.localized_name, i.name),
                CAST(i.cost AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER)
            FROM lakehouse.raw.mongo__purchase_events p
            INNER JOIN lakehouse.raw.pg__items i
                ON CAST(p.itemid AS INTEGER) = i.item_id
            WHERE p.match_id = {match_id}
              AND p.id IS NOT NULL
              AND i.cost > 2000

            UNION ALL

            SELECT
                CONCAT('streak:', CAST(s.id AS STRING)),
                LOWER(TRIM(s.type)),
                CAST(s.time AS INTEGER),
                CAST(s.steam_account_id AS BIGINT),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS INTEGER),
                CAST(NULL AS STRING),
                CAST(NULL AS INTEGER),
                CAST(s.heroId AS INTEGER),
                CAST(s.value AS INTEGER)
            FROM lakehouse.raw.mongo__streak_events s
            WHERE s.match_id = {match_id}
              AND s.id IS NOT NULL
              AND UPPER(TRIM(s.type)) IN (
                  'KILL_STREAK',
                  'MULTI_KILL'
              )
        )

        SELECT *
        FROM events
        ORDER BY
            event_time ASC NULLS LAST,
            event_id ASC
    """)

    events = [
        row.asDict(recursive=True)
        for row in data.toLocalIterator()
    ]

    heroes = spark.sql("""
        SELECT DISTINCT
            hero_id,
            localized_name
        FROM lakehouse.raw.pg__heroes
        WHERE hero_id IS NOT NULL
    """).collect()

    players = spark.sql(f"""
        SELECT DISTINCT
            account_id,
            hero_id,
            is_radiant
        FROM lakehouse.raw.pg__match_players
        WHERE match_id = {match_id}
          AND hero_id IS NOT NULL
    """).collect()


if not events:
    raise ValueError(f"Матч {match_id} не найден")


# hero_id -> название героя
hero_names = {
    int(row.hero_id): row.localized_name
    for row in heroes
    if row.localized_name is not None
}


# account_id -> hero_id
account_heroes = defaultdict(set)

# hero_id -> команда
hero_sides = defaultdict(set)

for row in players:
    hero_id = int(row.hero_id)

    if row.account_id is not None:
        account_heroes[int(row.account_id)].add(hero_id)

    if row.is_radiant is not None:
        hero_sides[hero_id].add(bool(row.is_radiant))


# Сопоставление аккаунтов с героями
account_to_hero = {}

for account_id, hero_ids in account_heroes.items():
    if account_id in (0, 4294967295):
        continue

    if len(hero_ids) == 1:
        hero_id = next(iter(hero_ids))

        if hero_id in hero_names:
            account_to_hero[account_id] = hero_names[hero_id]


# True = Radiant, False = Dire
hero_teams = {}

for hero_id, sides in hero_sides.items():
    if len(sides) == 1:
        hero_teams[hero_id] = next(iter(sides))


def get_hero_name(hero_id):
    if hero_id is None:
        return None

    return hero_names.get(int(hero_id))


def get_account_hero(account_id):
    if account_id is None:
        return None

    return account_to_hero.get(int(account_id))


def get_kill_from_radiant(attacker_id, target_id):
    if attacker_id is None or target_id is None:
        return None

    attacker_team = hero_teams.get(int(attacker_id))
    target_team = hero_teams.get(int(target_id))

    if attacker_team is None or target_team is None:
        return None

    if attacker_team == target_team:
        return None

    return 1 if attacker_team else 0


# Формируем события для LLM
clean_events = []

for event in events:
    event_type = event["event_type"]

    fact = {
        "event_id": event["event_id"],
        "type": event_type,
        "time": event["event_time"]
    }

    if event_type == "hero_kill":
        fact["attacker"] = (
            get_hero_name(event["attacker"])
            or event["attacker"]
        )

        fact["target"] = (
            get_hero_name(event["target"])
            or event["target"]
        )

        fact["kill_from_radiant"] = get_kill_from_radiant(
            event["attacker"],
            event["target"]
        )

        fact["killer_account_id"] = event["steam_account_id"]
        fact["gold"] = event["gold"]
        fact["xp"] = event["xp"]

    elif event_type == "tower_destroyed":
        fact["radiant_mask"] = event["radiant_mask"]
        fact["dire_mask"] = event["dire_mask"]

    elif event_type == "roshan_reward":
        fact["hero"] = get_account_hero(
            event["steam_account_id"]
        )
        fact["account_id"] = event["steam_account_id"]
        fact["gold"] = event["gold"]

    elif event_type == "buyback":
        fact["hero"] = get_account_hero(
            event["steam_account_id"]
        )
        fact["account_id"] = event["steam_account_id"]
        fact["cost"] = event["cost"]

    elif event_type == "item_purchase":
        fact["hero"] = get_account_hero(
            event["steam_account_id"]
        )

        fact["account_id"] = event["steam_account_id"]
        fact["item"] = event["item_name"]
        fact["item_cost"] = event["item_cost"]

    elif event_type in ("kill_streak", "multi_kill"):
        # В streak_events напрямую присутствует heroId
        hero_id = event["streak_hero_id"]

        fact["hero"] = (
            get_hero_name(hero_id)
            or hero_id
        )

        fact["account_id"] = event["steam_account_id"]
        fact["value"] = event["streak_value"]

    # Удаляем NULL, кроме kill_from_radiant
    fact = {
        key: value
        for key, value in fact.items()
        if value is not None or key == "kill_from_radiant"
    }

    clean_events.append(fact)


# Сортируем все события по игровому времени
clean_events.sort(
    key=lambda e: (
        e.get("time") is None,
        e.get("time") if e.get("time") is not None else float("inf"),
        e["event_id"]
    )
)


match_payload = {
    "match_id": match_id,
    "events": clean_events
}

match_json = json.dumps(
    match_payload,
    ensure_ascii=False,
    separators=(",", ":")
)

print("Событий всего:", len(clean_events))
print("Размер JSON:", len(match_json), "символов")

print(
    "Убийств Radiant:",
    sum(
        e.get("kill_from_radiant") == 1
        for e in clean_events
        if e["type"] == "hero_kill"
    )
)

print(
    "Убийств Dire:",
    sum(
        e.get("kill_from_radiant") == 0
        for e in clean_events
        if e["type"] == "hero_kill"
    )
)

print(
    "Покупок предметов дороже 2000 золота:",
    sum(
        e["type"] == "item_purchase"
        for e in clean_events
    )
)

print(
    "Событий KILL_STREAK:",
    sum(
        e["type"] == "kill_streak"
        for e in clean_events
    )
)

print(
    "Событий MULTI_KILL:",
    sum(
        e["type"] == "multi_kill"
        for e in clean_events
    )
)

print(
    json.dumps(
        match_payload,
        ensure_ascii=False,
        indent=2
    )
)


Событий всего: 181
Размер JSON: 29490 символов
Убийств Radiant: 25
Убийств Dire: 35
Покупок предметов дороже 2000 золота: 66
Событий KILL_STREAK: 16
Событий MULTI_KILL: 11
{
  "match_id": 9045752693,
  "events": [
    {
      "event_id": "kill:6af7fdde2652c8d2ed623041",
      "type": "hero_kill",
      "time": 15,
      "attacker": "Monkey King",
      "target": "Dragon Knight",
      "kill_from_radiant": 1,
      "killer_account_id": 107741011,
      "gold": 536,
      "xp": 66
    },
    {
      "event_id": "kill:6af7fdde2652c8d2ed62304d",
      "type": "hero_kill",
      "time": 218,
      "attacker": "Warlock",
      "target": "Queen of Pain",
      "kill_from_radiant": 1,
      "killer_account_id": 177076440,
      "gold": 314,
      "xp": 269
    },
    {
      "event_id": "kill:6af7fdde2652c8d2ed623068",
      "type": "hero_kill",
      "time": 233,
      "attacker": "Rubick",
      "target": "Shadow Fiend",
      "kill_from_radiant": 0,
      "killer_account_id": 341106989,
   

In [44]:
import openai
import os
import json

SYSTEM_PROMPT = """
Ты описываешь события матча Dota 2 на русском языке. Dota 2 - онлайн-игра в жанре MOBA.

На входе JSON одного матча. В массиве events факты уже расположены
по игровому времени. У каждого факта есть event_id, type и поля с данными. 
Поле time выражено в секундах. 
В event'ах с type=hero_kill поле kill_from_radiant = 1 означает, что убийство совершили игроки из команды Radiant, 0 - игроки из команды Dire. Учитывай количество значений kill_from_radiant, чтобы понять, какая команда доминировала по игре. 
Также в event есть type=item_purchase, который показывает покупки игроками ключевых предметов. Ещё есть события streak_events - это серии убийств, он является признаком доминации на карте. kill_streak и multi_kill — это события о достижении серии, а не дополнительные убийства. Учитывай это при написании общих итогов.

Нам нужно, чтобы ты составил общие итоги этого матча, основываясь на данных фактах. Общие итоги также пиши в хронологическом порядке. 

Объединяй близкие и повторяющиеся события. Например, три убийства
одного героя за первую минуту опиши одной фразой. Золото и опыт за отдельные убийства опускай.

Пиши как короткий послематчевый обзор. Сохраняй хронологию, имена
и подтверждённые числа. Если известно только время падения башни,
не добавляй её сторону, линию или уровень. Итог матча вынеси
в последний абзац, если он есть во входных данных.

Пиши в спокойном тоне, без агрессивных слов и выражений. Если в последнем event - событие tower_destroyed означает конец игры и дальше по временным отрезкам идти не надо. 
После него нужно плавно завершить саммари. Не упоминай конкретно mask вышек. Также не упоминай конкретно временные отрезки в числах.

Если первое игровое событие — первая кровь, начни первый абзац
с «Игра началась с…» и упомяни её один раз. 
"""

import json

# match_json = json.dumps(match_payload, ensure_ascii=False)

client = openai.OpenAI(
    api_key="sk-67qOmCQbXkp39cHsUJboMw",
    base_url="https://api.vsellm.ru/v1"
)

response = client.chat.completions.create(
    model="openai/gpt-5.4-nano",
    messages=[
        {"role": "system", "content": f"{SYSTEM_PROMPT}"},
        {"role": "user", "content": json.dumps(match_json, ensure_ascii=False)}
    ]
)

print(response.choices[0].message.content)

Игра началась с первой крови: Monkey King на 15-й минуте (по таймкоду события) убил Dragon Knight.

Дальше на линии и в небольших стычках команды обменивались убийствами: Monkey King и Warlock/Queen of Pain дали несколько точечных разменов, затем Witch Doctor разогнался, оформив серию и взяв сразу несколько целей подряд. Вскоре Rubick перехватил инициативу в центре — в районе середины игры он несколько раз наказал Shadow Fiend и стал наращивать преимущество по темпу.

По покупкам ключевых предметов выделялись ранние артефакты Rubick (Phylactery, Boots of Travel), Monkey King (Diffusal Blade, позже BKB) и Witch Doctor (Aether Lens, Boots of Travel). В то же время Dire активно отвечала через контр-выходы по киллам и укрепления состава (например, Black King Bar у Queen of Pain и прогресс на контроле/выживании).

Ближе к середине и дальше пошли разрушения вышек: несколько объектов пали по обе стороны, но Dire регулярно закреплялась по карте через последовательные атаки. Это сопровождалось 